# Take dates

First, we'll name the folder containing our files.

In [33]:
root_dir = '/Users/shiva/WBS/case08_Azure/Notebooks/resource/'
source = root_dir+'/resource/silver'

In [32]:
source = '/Users/shiva/WBS/case08_Azure/Notebooks/resource/silver/'

Most straightforward and familiar to us way is using pandas. We can read the files and use `.min()` and `.max()` to find the least and greatest values in the `"timestamp"` columns.

In [9]:
for name, df in data.items():
    print(f"\n{name}:")
    print("Columns:", df.columns.tolist())
    print(df.head(3))


flow:
Columns: ['flow_out', 'flow_in']
   flow_out  flow_in
0         0        0
1         0        0
2         0        0

humidity:
Columns: ['timestamp', 'humidity']
                  timestamp  humidity
0 2017-01-01 12:00:00+00:00     98.04
1 2017-01-02 00:00:00+00:00     98.61
2 2017-01-02 12:00:00+00:00     99.00

temperature:
Columns: ['timestamp', 'temperature']
                  timestamp  temperature
0 2017-01-01 13:10:00+00:00          NaN
1 2017-01-01 13:15:00+00:00        12.34
2 2017-01-01 13:20:00+00:00        12.27

weight:
Columns: ['timestamp', 'weight']
                  timestamp    weight
0 2017-01-01 12:00:00+00:00  50736.79
1 2017-01-02 00:00:00+00:00  50700.68
2 2017-01-02 12:00:00+00:00  50614.91


In [10]:
print("Index name:", flow.index.name)
print("Index type:", type(flow.index))
print("Index sample:", flow.index[:5])
print(flow.shape)

Index name: None
Index type: <class 'pandas.RangeIndex'>
Index sample: RangeIndex(start=0, stop=5, step=1)
(1256918, 2)


In [13]:
for name, df in data.items():
    print(name, df["timestamp"].dtype)

flow datetime64[us, UTC]
humidity datetime64[us, UTC]
temperature datetime64[us]
weight datetime64[us, UTC]


In [17]:
import pandas as pd

flow = pd.read_parquet(source+"flow/schwartau__2026-10-14T21h25m37s.parquet")
humidity = pd.read_parquet(source+"humidity/schwartau__2026-10-14T21h25m37s.parquet")
temperature = pd.read_parquet(source+"temperature/schwartau__2026-10-14T21h25m37s.parquet")
weight = pd.read_parquet(source+"weight/schwartau__2026-10-14T21h25m37s.parquet")

start_date = min(flow["timestamp"].min(), humidity["timestamp"].min(), temperature["timestamp"].min(), weight["timestamp"].min())
end_date = max(flow["timestamp"].max(), humidity["timestamp"].max(), temperature["timestamp"].max(), weight["timestamp"].max())

print("Start date:", start_date)
print("End date:", end_date)

Start date: 2017-01-01 12:00:00+00:00
End date: 2019-05-31 14:15:00+00:00


# Weather

## 1. Make a request

In [18]:
import requests

url = "https://api.brightsky.dev/weather"

querystring = {"date":"2017-01-01 12:00:00+0000",
               "lat":"53.919444",
               "lon":"10.6975",
               "max_dist":"20000",
               "tz":"UTC",
               "units":"dwd"}

headers = {"Accept": "application/json"}

response = requests.get(url, headers=headers, params=querystring)

print(response.status_code)

200


`.status_code == 200`, so we can continue with processing data.

In [21]:
coords = {
    "schwartau": {"lat": 53.919444, "lon": 10.6975}
}

In [22]:
coords['schwartau']

{'lat': 53.919444, 'lon': 10.6975}

In [23]:
min_date = {"date": start_date}

In [24]:
coords['schwartau'] | min_date

{'lat': 53.919444,
 'lon': 10.6975,
 'date': Timestamp('2017-01-01 12:00:00+0000', tz='UTC')}

In [25]:
url = "https://api.brightsky.dev/weather"

querystring = coords['schwartau'] | {"date": start_date,
                                     "max_dist":"20000",
                                     "tz":"UTC",
                                     "units":"dwd"}

headers = {"Accept": "application/json"}

response = requests.get(url, headers=headers, params=querystring)

print(response.status_code)

200


`.status_code == 200`, so we can continue with processing data.

In [26]:
# transform to JSON (nested dictionary)
weather = response.json()

In [27]:
response.url

'https://api.brightsky.dev/weather?lat=53.919444&lon=10.6975&date=2017-01-01+12%3A00%3A00%2B00%3A00&max_dist=20000&tz=UTC&units=dwd'

In [28]:
weather

{'weather': [{'timestamp': '2017-01-01T12:00:00+00:00',
   'source_id': 286541,
   'precipitation': 0.0,
   'pressure_msl': 1013.8,
   'sunshine': 0,
   'temperature': 5.2,
   'wind_direction': 230,
   'wind_speed': 10.8,
   'cloud_cover': 100,
   'dew_point': 3.5,
   'relative_humidity': 89,
   'visibility': 5450,
   'wind_gust_direction': 240,
   'wind_gust_speed': 26.6,
   'condition': 'dry',
   'precipitation_probability': None,
   'precipitation_probability_6h': None,
   'solar': 0.036,
   'fallback_source_ids': {'relative_humidity': 7102,
    'wind_direction': 7102,
    'wind_speed': 7102,
    'temperature': 7102,
    'sunshine': 7102,
    'wind_gust_direction': 7102,
    'pressure_msl': 7102,
    'visibility': 7102,
    'solar': 7102,
    'dew_point': 7102,
    'cloud_cover': 7102,
    'wind_gust_speed': 7102},
   'icon': 'cloudy'},
  {'timestamp': '2017-01-01T13:00:00+00:00',
   'source_id': 286541,
   'precipitation': 0.0,
   'pressure_msl': 1013.4,
   'sunshine': 0,
   'tempe

In [1]:
# import json
# from IPython.display import HTML

# def render_json_tree(data, title="JSON Data"):
#     """Renders a collapsible JSON tree viewer compatible with VS Code."""

#     def build_tree(obj):
#         if isinstance(obj, dict):
#             items = "".join(
#                 f"<li style='margin-left: 15px;'><strong>{k}</strong>: {build_tree(v)}</li>"
#                 for k, v in obj.items()
#             )
#             return f"<details style='cursor: pointer;' open><summary style='color: #4EC9B0;'>dict ({len(obj)})</summary><ul style='list-style-type: none; padding-left: 10px; margin: 2px 0;'>{items}</ul></details>"
#         elif isinstance(obj, list):
#             items = "".join(
#                 f"<li style='margin-left: 15px;'>[{i}]: {build_tree(item)}</li>"
#                 for i, item in enumerate(obj)
#             )
#             return f"<details style='cursor: pointer;' open><summary style='color: #CE9178;'>list ({len(obj)})</summary><ul style='list-style-type: none; padding-left: 10px; margin: 2px 0;'>{items}</ul></details>"
#         else:
#             return f"<span style='color: #9CDCFFE;'>{json.dumps(obj)}</span>"

#     html_content = f"""
#     <div style="font-family: monospace; font-size: 13px; line-height: 1.4; padding: 10px; border: 1px solid #444; border-radius: 6px;">
#         <strong style="font-size: 14px; color: #569CD6;">{title}</strong>
#         {build_tree(data)}
#     </div>
#     """
#     return HTML(html_content)

# # Usage:
# # render_json_tree(weather, title="Weather API Response")

In [2]:
# # view JSON data in a structured, collapsible “tree” viewer in Jupyter lab
# import json
# from IPython.display import JSON

# # JSON(weather)

In [35]:
# write raw data to file
import json

weather_sink = root_dir+"/bronze/archive/weather/"

with open (weather_sink+f"schwartau__{start_date.strftime('%Y-%m-%d')}.json", "w") as f:
    json.dump(weather, f, indent=4)

## Explore and create a df

In [36]:
# check what keys are in the data
weather.keys()

dict_keys(['weather', 'sources'])

### 2.1 "weather" key

In [37]:
# what does the first value look like?
weather["weather"]

[{'timestamp': '2017-01-01T12:00:00+00:00',
  'source_id': 286541,
  'precipitation': 0.0,
  'pressure_msl': 1013.8,
  'sunshine': 0,
  'temperature': 5.2,
  'wind_direction': 230,
  'wind_speed': 10.8,
  'cloud_cover': 100,
  'dew_point': 3.5,
  'relative_humidity': 89,
  'visibility': 5450,
  'wind_gust_direction': 240,
  'wind_gust_speed': 26.6,
  'condition': 'dry',
  'precipitation_probability': None,
  'precipitation_probability_6h': None,
  'solar': 0.036,
  'fallback_source_ids': {'relative_humidity': 7102,
   'wind_direction': 7102,
   'wind_speed': 7102,
   'temperature': 7102,
   'sunshine': 7102,
   'wind_gust_direction': 7102,
   'pressure_msl': 7102,
   'visibility': 7102,
   'solar': 7102,
   'dew_point': 7102,
   'cloud_cover': 7102,
   'wind_gust_speed': 7102},
  'icon': 'cloudy'},
 {'timestamp': '2017-01-01T13:00:00+00:00',
  'source_id': 286541,
  'precipitation': 0.0,
  'pressure_msl': 1013.4,
  'sunshine': 0,
  'temperature': 5.0,
  'wind_direction': 230,
  'wind_s

In [38]:
# can we convert directly to DataFrame?
pd.json_normalize(weather["weather"])#['source_id'].value_counts()

,timestamp,source_id,precipitation,pressure_msl,sunshine,temperature,wind_direction,wind_speed,cloud_cover,dew_point,...,fallback_source_ids.wind_speed,fallback_source_ids.temperature,fallback_source_ids.sunshine,fallback_source_ids.wind_gust_direction,fallback_source_ids.pressure_msl,fallback_source_ids.visibility,fallback_source_ids.solar,fallback_source_ids.dew_point,fallback_source_ids.cloud_cover,fallback_source_ids.wind_gust_speed
0,2017-01-01T12:00:00+00:00,286541,0.0,1013.8,0.0,5.2,230,10.8,100,3.5,...,7102,7102,7102.0,7102,7102,7102.0,7102,7102,7102,7102
1,2017-01-01T13:00:00+00:00,286541,0.0,1013.4,0.0,5.0,230,10.8,87,3.2,...,7102,7102,7102.0,7102,7102,7102.0,7102,7102,7102,7102
2,2017-01-01T14:00:00+00:00,286541,0.0,1013.4,0.0,4.6,230,12.6,87,2.9,...,7102,7102,7102.0,7102,7102,7102.0,7102,7102,7102,7102
3,2017-01-01T15:00:00+00:00,286541,0.0,1013.2,0.0,4.2,230,11.9,100,2.6,...,7102,7102,7102.0,7102,7102,7102.0,7102,7102,7102,7102
4,2017-01-01T16:00:00+00:00,286541,0.0,1013.2,0.0,3.8,240,11.9,100,2.3,...,7102,7102,7102.0,7102,7102,7102.0,7102,7102,7102,7102
5,2017-01-01T17:00:00+00:00,286541,0.0,1013.4,0.0,3.6,240,10.1,87,2.1,...,7102,7102,7102.0,7102,7102,7102.0,7102,7102,7102,7102
6,2017-01-01T18:00:00+00:00,286541,0.0,1013.3,0.0,3.7,250,11.5,100,1.9,...,7102,7102,7102.0,7102,7102,7102.0,7102,7102,7102,7102
7,2017-01-01T19:00:00+00:00,286541,0.3,1013.3,0.0,3.6,250,15.5,87,2.1,...,7102,7102,7102.0,7102,7102,7102.0,7102,7102,7102,7102
8,2017-01-01T20:00:00+00:00,286541,0.3,1013.5,0.0,2.9,250,15.5,87,2.2,...,7102,7102,7102.0,7102,7102,7102.0,7102,7102,7102,7102
9,2017-01-01T21:00:00+00:00,286541,0.3,1013.8,NaN,2.4,260,15.1,87,1.6,...,7102,7102,NaN,7102,7102,7102.0,7102,7102,7102,7102


### 2.2 "sources" key

In [39]:
weather["sources"]

[{'id': 286541,
  'dwd_station_id': '04602',
  'observation_type': 'historical',
  'lat': 53.9385,
  'lon': 10.6983,
  'height': 27.0,
  'station_name': 'Schwartau,Bad -Groß Parin',
  'wmo_station_id': 'A791',
  'first_record': '2010-01-01T00:00:00+00:00',
  'last_record': '2025-02-25T23:00:00+00:00',
  'distance': 2122},
 {'id': 7102,
  'dwd_station_id': '03086',
  'observation_type': 'historical',
  'lat': 53.8025,
  'lon': 10.6989,
  'height': 14.74,
  'station_name': 'Lübeck-Blankensee',
  'wmo_station_id': '10156',
  'first_record': '2010-01-01T00:00:00+00:00',
  'last_record': '2026-10-13T23:00:00+00:00',
  'distance': 13018}]

In [40]:
sources_df = pd.DataFrame(weather["sources"])
sources_df

,id,dwd_station_id,observation_type,lat,lon,height,station_name,wmo_station_id,first_record,last_record,distance
0,286541,04602,historical,53.9385,10.6983,27.00,"Schwartau,Bad -Groß Parin",A791,2010-01-01T00:00:00+00:00,2025-02-25T23:00:00+00:00,2122
1,7102,03086,historical,53.8025,10.6989,14.74,Lübeck-Blankensee,10156,2010-01-01T00:00:00+00:00,2026-10-13T23:00:00+00:00,13018


In [41]:
weather_df = pd.json_normalize(weather["weather"])
weather_df.head()

,timestamp,source_id,precipitation,pressure_msl,sunshine,temperature,wind_direction,wind_speed,cloud_cover,dew_point,...,fallback_source_ids.wind_speed,fallback_source_ids.temperature,fallback_source_ids.sunshine,fallback_source_ids.wind_gust_direction,fallback_source_ids.pressure_msl,fallback_source_ids.visibility,fallback_source_ids.solar,fallback_source_ids.dew_point,fallback_source_ids.cloud_cover,fallback_source_ids.wind_gust_speed
0,2017-01-01T12:00:00+00:00,286541,0.0,1013.8,0.0,5.2,230,10.8,100,3.5,...,7102,7102,7102.0,7102,7102,7102.0,7102,7102,7102,7102
1,2017-01-01T13:00:00+00:00,286541,0.0,1013.4,0.0,5.0,230,10.8,87,3.2,...,7102,7102,7102.0,7102,7102,7102.0,7102,7102,7102,7102
2,2017-01-01T14:00:00+00:00,286541,0.0,1013.4,0.0,4.6,230,12.6,87,2.9,...,7102,7102,7102.0,7102,7102,7102.0,7102,7102,7102,7102
3,2017-01-01T15:00:00+00:00,286541,0.0,1013.2,0.0,4.2,230,11.9,100,2.6,...,7102,7102,7102.0,7102,7102,7102.0,7102,7102,7102,7102
4,2017-01-01T16:00:00+00:00,286541,0.0,1013.2,0.0,3.8,240,11.9,100,2.3,...,7102,7102,7102.0,7102,7102,7102.0,7102,7102,7102,7102


### 2.4 What about those fallback ids?

If we start to look into these fallback source ids, it is a bit of a mess. Each row (timestamp) might have different sources for different measures. Our above approach won't work then: a single row would need to identify several weather stations.

Perhaps better then, to ignore most station features and focus on one: how far is the station from our hives? We can then note individually for each measure how distant the measure was taken from the hive. Because this can result in a different answer for each measure at each row, we're better off modifying the dictionary directly (pandas DataFrames are optimized for column operations, not cell-by-cell).

In [42]:
weather_df.head()

,timestamp,source_id,precipitation,pressure_msl,sunshine,temperature,wind_direction,wind_speed,cloud_cover,dew_point,...,fallback_source_ids.wind_speed,fallback_source_ids.temperature,fallback_source_ids.sunshine,fallback_source_ids.wind_gust_direction,fallback_source_ids.pressure_msl,fallback_source_ids.visibility,fallback_source_ids.solar,fallback_source_ids.dew_point,fallback_source_ids.cloud_cover,fallback_source_ids.wind_gust_speed
0,2017-01-01T12:00:00+00:00,286541,0.0,1013.8,0.0,5.2,230,10.8,100,3.5,...,7102,7102,7102.0,7102,7102,7102.0,7102,7102,7102,7102
1,2017-01-01T13:00:00+00:00,286541,0.0,1013.4,0.0,5.0,230,10.8,87,3.2,...,7102,7102,7102.0,7102,7102,7102.0,7102,7102,7102,7102
2,2017-01-01T14:00:00+00:00,286541,0.0,1013.4,0.0,4.6,230,12.6,87,2.9,...,7102,7102,7102.0,7102,7102,7102.0,7102,7102,7102,7102
3,2017-01-01T15:00:00+00:00,286541,0.0,1013.2,0.0,4.2,230,11.9,100,2.6,...,7102,7102,7102.0,7102,7102,7102.0,7102,7102,7102,7102
4,2017-01-01T16:00:00+00:00,286541,0.0,1013.2,0.0,3.8,240,11.9,100,2.3,...,7102,7102,7102.0,7102,7102,7102.0,7102,7102,7102,7102


In [43]:
weather["weather"]

[{'timestamp': '2017-01-01T12:00:00+00:00',
  'source_id': 286541,
  'precipitation': 0.0,
  'pressure_msl': 1013.8,
  'sunshine': 0,
  'temperature': 5.2,
  'wind_direction': 230,
  'wind_speed': 10.8,
  'cloud_cover': 100,
  'dew_point': 3.5,
  'relative_humidity': 89,
  'visibility': 5450,
  'wind_gust_direction': 240,
  'wind_gust_speed': 26.6,
  'condition': 'dry',
  'precipitation_probability': None,
  'precipitation_probability_6h': None,
  'solar': 0.036,
  'fallback_source_ids': {'relative_humidity': 7102,
   'wind_direction': 7102,
   'wind_speed': 7102,
   'temperature': 7102,
   'sunshine': 7102,
   'wind_gust_direction': 7102,
   'pressure_msl': 7102,
   'visibility': 7102,
   'solar': 7102,
   'dew_point': 7102,
   'cloud_cover': 7102,
   'wind_gust_speed': 7102},
  'icon': 'cloudy'},
 {'timestamp': '2017-01-01T13:00:00+00:00',
  'source_id': 286541,
  'precipitation': 0.0,
  'pressure_msl': 1013.4,
  'sunshine': 0,
  'temperature': 5.0,
  'wind_direction': 230,
  'wind_s

In [44]:
sources_df = pd.DataFrame(weather["sources"])

# these parts of the returned data are not actual measurements
measures_to_ignore = ["timestamp",
                       "source_id",
                       "condition",
                       "precipitation_probability", # for forecasts only, and we use historical data
                       "precipitation_probability_6h", # for forecasts only, and we use historical data
                       "icon",
                       "fallback_source_ids"]

for time in weather["weather"]:
    temp_dict = {}
    for measure in time:
        if measure in measures_to_ignore:
            continue

        # if the measure appears in "fallback_source_ids", it has a different source than the main
        if measure in time["fallback_source_ids"]:
            temp_dict[f"{measure}_source_distance"] = sources_df.loc[sources_df["id"] == time["fallback_source_ids"][measure], "distance"].values[0]
        else:
            temp_dict[f"{measure}_source_distance"] = sources_df.loc[sources_df["id"] == time["source_id"], "distance"].values[0]

    # update dictionary at one timestamp
    time.update(temp_dict)

weather

{'weather': [{'timestamp': '2017-01-01T12:00:00+00:00',
   'source_id': 286541,
   'precipitation': 0.0,
   'pressure_msl': 1013.8,
   'sunshine': 0,
   'temperature': 5.2,
   'wind_direction': 230,
   'wind_speed': 10.8,
   'cloud_cover': 100,
   'dew_point': 3.5,
   'relative_humidity': 89,
   'visibility': 5450,
   'wind_gust_direction': 240,
   'wind_gust_speed': 26.6,
   'condition': 'dry',
   'precipitation_probability': None,
   'precipitation_probability_6h': None,
   'solar': 0.036,
   'fallback_source_ids': {'relative_humidity': 7102,
    'wind_direction': 7102,
    'wind_speed': 7102,
    'temperature': 7102,
    'sunshine': 7102,
    'wind_gust_direction': 7102,
    'pressure_msl': 7102,
    'visibility': 7102,
    'solar': 7102,
    'dew_point': 7102,
    'cloud_cover': 7102,
    'wind_gust_speed': 7102},
   'icon': 'cloudy',
   'precipitation_source_distance': np.int64(2122),
   'pressure_msl_source_distance': np.int64(13018),
   'sunshine_source_distance': np.int64(13018

## 3. A few days

In [45]:
import requests
import json
import pandas as pd
from datetime import timedelta

weather_sink = root_dir+"bronze/archive/"

coords = {
    "schwartau": {"lat": 53.919444, "lon": 10.6975}
}
measures_to_ignore = ["timestamp", "source_id", "condition", "precipitation_probability",
                      "precipitation_probability_6h", "icon", "fallback_source_ids"]

url = "https://api.brightsky.dev/weather"
headers = {"Accept": "application/json"}
params = coords["schwartau"] | {"date": start_date, "last_date": start_date+timedelta(days=3)}

response = requests.get(url, headers=headers, params=params)

if response.status_code == 200:
    weather = response.json()
    with open (weather_sink+f"schwartau__{start_date.strftime('%Y-%m-%d')}-{(start_date+timedelta(days=3)).strftime('%Y-%m-%d')}.json", "w") as f:
        json.dump(weather, f, indent=4)

    sources_df = pd.DataFrame(weather["sources"])
    for time in weather["weather"]:
        temp_dict = {}
        for measure in time.keys():
            if measure in measures_to_ignore:
                continue
            if measure in time["fallback_source_ids"]:
                temp_dict[f"{measure}_source_distance"] = sources_df.loc[sources_df["id"] == time["fallback_source_ids"][measure], "distance"].values[0]
            else:
                temp_dict[f"{measure}_source_distance"] = sources_df.loc[sources_df["id"] == time["source_id"], "distance"].values[0]
        time.update(temp_dict)

    weather_df = (
        pd.DataFrame(weather["weather"])
        .drop(["source_id", "visibility", "condition", "icon", "precipitation_probability", "precipitation_probability_6h", "fallback_source_ids"], axis=1)
    )
    weather_df["timestamp"] = pd.to_datetime(weather_df["timestamp"])
else:
    print(f"Failure to gather weather data for period {start_date} to {start_date+timedelta(days=3)}.\nReason: {response.text}")

weather_df

,timestamp,precipitation,pressure_msl,sunshine,temperature,wind_direction,wind_speed,cloud_cover,dew_point,relative_humidity,...,temperature_source_distance,wind_direction_source_distance,wind_speed_source_distance,cloud_cover_source_distance,dew_point_source_distance,relative_humidity_source_distance,visibility_source_distance,wind_gust_direction_source_distance,wind_gust_speed_source_distance,solar_source_distance
0,2017-01-01 12:00:00+00:00,0.0,1013.8,0.0,5.2,230,10.8,100,3.5,89,...,13018,13018,13018,13018,13018,13018,13018,13018,13018,13018
1,2017-01-01 13:00:00+00:00,0.0,1013.4,0.0,5.0,230,10.8,87,3.2,88,...,13018,13018,13018,13018,13018,13018,13018,13018,13018,13018
2,2017-01-01 14:00:00+00:00,0.0,1013.4,0.0,4.6,230,12.6,87,2.9,89,...,13018,13018,13018,13018,13018,13018,13018,13018,13018,13018
3,2017-01-01 15:00:00+00:00,0.0,1013.2,0.0,4.2,230,11.9,100,2.6,89,...,13018,13018,13018,13018,13018,13018,13018,13018,13018,13018
4,2017-01-01 16:00:00+00:00,0.0,1013.2,0.0,3.8,240,11.9,100,2.3,90,...,13018,13018,13018,13018,13018,13018,13018,13018,13018,13018
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
68,2017-01-04 08:00:00+00:00,0.0,997.8,0.0,4.9,280,31.0,87,-0.1,70,...,13018,13018,13018,13018,13018,13018,13018,13018,13018,13018
69,2017-01-04 09:00:00+00:00,0.2,998.5,0.0,4.3,290,29.9,87,1.3,81,...,13018,13018,13018,13018,13018,13018,13018,13018,13018,13018
70,2017-01-04 10:00:00+00:00,0.0,999.5,4.0,3.4,290,25.9,37,0.9,84,...,13018,13018,13018,13018,13018,13018,13018,13018,13018,13018
71,2017-01-04 11:00:00+00:00,0.0,1000.3,59.0,4.6,290,23.0,0,1.3,79,...,13018,13018,13018,13018,13018,13018,13018,13018,13018,13018


In [46]:
weather_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 73 entries, 0 to 72
Data columns (total 26 columns):
 #   Column                               Non-Null Count  Dtype              
---  ------                               --------------  -----              
 0   timestamp                            73 non-null     datetime64[us, UTC]
 1   precipitation                        73 non-null     float64            
 2   pressure_msl                         73 non-null     float64            
 3   sunshine                             55 non-null     float64            
 4   temperature                          73 non-null     float64            
 5   wind_direction                       73 non-null     int64              
 6   wind_speed                           73 non-null     float64            
 7   cloud_cover                          73 non-null     int64              
 8   dew_point                            73 non-null     float64            
 9   relative_humidity                    73 non-n

## 4. All dates

In [47]:
# Soluiton 1: with source distance

import requests
import json
import pandas as pd
#from datetime import timedelta

raw_sink = root_dir+"bronze/archive/weather/"

coords = {
    "schwartau": {"lat": 53.919444, "lon": 10.6975}
}
measures_to_ignore = ["timestamp", "source_id", "condition", "precipitation_probability",
                      "precipitation_probability_6h", "icon", "fallback_source_ids"]

url = "https://api.brightsky.dev/weather"
headers = {"Accept": "application/json"}
# params = coords["schwartau"] | {"date": start_date, "last_date": end_date}

params = {"lat": 53.919444, "lon": 10.6975, "date": start_date, "last_date": end_date}

response = requests.get(url, headers=headers, params=params)

if response.status_code == 200:
    weather = response.json()

    # 1st export the json response to bronze/archive/weather
    with open (raw_sink+f"schwartau__{start_date.strftime('%Y-%m-%d')}-{end_date.strftime('%Y-%m-%d')}.json", "w") as f:
        json.dump(weather, f, indent=4)

    sources_df = pd.DataFrame(weather["sources"])

    # getting the distance from which the api got every measure
    for time in weather["weather"]:
        temp_dict = {}
        for measure in time.keys():
            if measure in measures_to_ignore:
                continue
            if measure in time.get("fallback_source_ids",{}):
                temp_dict[f"{measure}_source_distance"] = sources_df.loc[sources_df["id"] == time.get("fallback_source_ids",{}).get(measure), "distance"].values[0]
            else:
                temp_dict[f"{measure}_source_distance"] = sources_df.loc[sources_df["id"] == time["source_id"], "distance"].values[0]
        time.update(temp_dict)

    weather_df = (
        pd.DataFrame(weather["weather"])
        .drop(["source_id", "visibility", "condition", "icon", "precipitation_probability", "precipitation_probability_6h", "fallback_source_ids"], axis=1)
    )
    weather_df["timestamp"] = pd.to_datetime(weather_df["timestamp"])
else:
    print(f"Failure to gather weather data for period {start_date} to {end_date}.\nReason: {response.text}")

weather_df.info()

KeyboardInterrupt: 

In [48]:
# Soluiton 2: without source distance

import requests
import json
import pandas as pd

raw_sink = root_dir + "bronze/archive/weather/"

url = "https://api.brightsky.dev/weather"
headers = {"Accept": "application/json"}
params = {"lat": 53.919444, "lon": 10.6975, "date": start_date, "last_date": end_date}

response = requests.get(url, headers=headers, params=params)

if response.status_code == 200:
    weather = response.json()

    # export the json response to bronze/archive/weather
    with open(raw_sink + f"schwartau__{start_date.strftime('%Y-%m-%d')}-{end_date.strftime('%Y-%m-%d')}.json", "w") as f:
        json.dump(weather, f, indent=4)

    weather_df = (
        pd.DataFrame(weather["weather"])
        .drop(columns=["source_id", "visibility", "condition", "icon",
                       "precipitation_probability", "precipitation_probability_6h",
                       "fallback_source_ids"],
              errors="ignore")
    )
    weather_df["timestamp"] = pd.to_datetime(weather_df["timestamp"])
else:
    print(f"Failure to gather weather data for period {start_date} to {end_date}.\nReason: {response.text}")

weather_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 21123 entries, 0 to 21122
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype              
---  ------               --------------  -----              
 0   timestamp            21123 non-null  datetime64[us, UTC]
 1   precipitation        21123 non-null  float64            
 2   pressure_msl         21123 non-null  float64            
 3   sunshine             15843 non-null  float64            
 4   temperature          21123 non-null  float64            
 5   wind_direction       21123 non-null  int64              
 6   wind_speed           21123 non-null  float64            
 7   cloud_cover          21109 non-null  float64            
 8   dew_point            21123 non-null  float64            
 9   relative_humidity    21123 non-null  int64              
 10  wind_gust_direction  21123 non-null  int64              
 11  wind_gust_speed      21123 non-null  float64            
 12  solar                21123 no

In [49]:
weather_df

,timestamp,precipitation,pressure_msl,sunshine,temperature,wind_direction,wind_speed,cloud_cover,dew_point,relative_humidity,wind_gust_direction,wind_gust_speed,solar
0,2017-01-01 12:00:00+00:00,0.0,1013.8,0.0,5.2,230,10.8,100.0,3.5,89,240,26.6,0.036
1,2017-01-01 13:00:00+00:00,0.0,1013.4,0.0,5.0,230,10.8,87.0,3.2,88,240,26.3,0.050
2,2017-01-01 14:00:00+00:00,0.0,1013.4,0.0,4.6,230,12.6,87.0,2.9,89,240,31.7,0.037
3,2017-01-01 15:00:00+00:00,0.0,1013.2,0.0,4.2,230,11.9,100.0,2.6,89,250,29.2,0.007
4,2017-01-01 16:00:00+00:00,0.0,1013.2,0.0,3.8,240,11.9,100.0,2.3,90,250,26.3,0.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
21118,2019-05-31 10:00:00+00:00,0.0,1021.7,31.0,20.4,250,14.4,100.0,14.3,68,240,34.9,0.520
21119,2019-05-31 11:00:00+00:00,0.0,1021.6,30.0,21.7,260,18.0,100.0,14.7,64,270,33.8,0.568
21120,2019-05-31 12:00:00+00:00,0.0,1021.8,10.0,20.6,270,16.6,100.0,14.1,66,240,32.4,0.328
21121,2019-05-31 13:00:00+00:00,0.0,1022.0,0.0,20.8,270,14.0,87.0,14.5,67,300,27.4,0.204


In [50]:
weather_df.isna().sum()

timestamp                 0
precipitation             0
pressure_msl              0
sunshine               5280
temperature               0
wind_direction            0
wind_speed                0
cloud_cover              14
dew_point                 0
relative_humidity         0
wind_gust_direction       0
wind_gust_speed           0
solar                     0
dtype: int64

# Write data

In [51]:
# 2nd Export

processed_sink = root_dir+"/bronze/archive/weather/"
weather_df.to_parquet(processed_sink+f"schwartau__{start_date.strftime('%Y-%m-%d')}-{end_date.strftime('%Y-%m-%d')}.parquet")

In [52]:
import sys
sys.version

'3.13.13 | packaged by Anaconda, Inc. | (main, Apr 14 2026, 06:12:50) [MSC v.1942 64 bit (AMD64)]'

In [53]:
import requests
my_url_w_endpoint = 'https://api.brightsky.dev/weather'
my_params = {'lat': 50,
             'lon': 11,
             'date': '2026-09-29',
             'last_date': '2026-09-30',
             'max_dist': 20000}
response = requests.get(url = my_url_w_endpoint, params = my_params)
print(response.url)
print(response.status_code)
response.json()

https://api.brightsky.dev/weather?lat=50&lon=11&date=2026-09-29&last_date=2026-09-30&max_dist=20000
200


{'weather': [{'timestamp': '2026-09-29T00:00:00+00:00',
   'source_id': 96874,
   'precipitation': 0.0,
   'pressure_msl': 1022.8,
   'sunshine': None,
   'temperature': 11.6,
   'wind_direction': 30,
   'wind_speed': 2.5,
   'cloud_cover': 62,
   'dew_point': 9.8,
   'relative_humidity': 89,
   'visibility': 33890,
   'wind_gust_direction': 10,
   'wind_gust_speed': 7.2,
   'condition': 'dry',
   'precipitation_probability': None,
   'precipitation_probability_6h': None,
   'solar': None,
   'fallback_source_ids': {'dew_point': 222290,
    'temperature': 222290,
    'relative_humidity': 222290,
    'cloud_cover': 6882,
    'visibility': 6882,
    'wind_gust_speed': 6882,
    'wind_gust_direction': 6882,
    'wind_direction': 6882,
    'wind_speed': 6882,
    'pressure_msl': 6882},
   'icon': 'partly-cloudy-night'},
  {'timestamp': '2026-09-29T01:00:00+00:00',
   'source_id': 96874,
   'precipitation': 0.0,
   'pressure_msl': 1023.0,
   'sunshine': None,
   'temperature': 10.7,
   'win